In [1]:

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")

Libraries imported successfully!
Pandas version: 2.1.4


In [2]:

RAW_PATH = Path("../dataset/raw/")
PROCESSED_PATH = Path("../dataset/processed/")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Raw data path:       {RAW_PATH.absolute()}")
print(f"Processed data path: {PROCESSED_PATH.absolute()}")

Raw data path:       e:\Clash Royale\notebooks\..\dataset\raw
Processed data path: e:\Clash Royale\notebooks\..\dataset\processed


In [3]:

cards_df = pd.read_csv(RAW_PATH / "cards.csv")
decks_df = pd.read_csv(RAW_PATH / "decks.csv")
battles_df = pd.read_csv(RAW_PATH / "battles.csv")
meta_df = pd.read_csv(RAW_PATH / "meta_stats.csv")

print("All raw datasets loaded!\n")
print(f"Cards:    {cards_df.shape}")
print(f"Decks:    {decks_df.shape}")
print(f"Battles:  {battles_df.shape}")
print(f"Meta:     {meta_df.shape}")

All raw datasets loaded!

Cards:    (127, 12)
Decks:    (480, 17)
Battles:  (5000, 12)
Meta:     (635, 6)


In [4]:

print("MISSING VALUES (BEFORE CLEANING)")
print("=" * 50)

datasets = {
    "Cards":   cards_df,
    "Decks":   decks_df,
    "Battles": battles_df,
    "Meta":    meta_df
}

for name, df in datasets.items():
    total_missing = df.isnull().sum().sum()
    print(f"{name:10} → Missing: {total_missing}")

MISSING VALUES (BEFORE CLEANING)
Cards      → Missing: 0
Decks      → Missing: 0
Battles    → Missing: 0
Meta       → Missing: 0


In [5]:

print("CLEANING CARDS DATASET")
print("=" * 50)

initial_rows = len(cards_df)

cards_df = cards_df.drop_duplicates(subset=['name'])
print(f"Removed duplicate cards: {initial_rows - len(cards_df)}")

cards_df = cards_df.dropna()
print(f"Dropped rows with NaN values")


cards_df['name'] = cards_df['name'].str.strip()
cards_df['rarity'] = cards_df['rarity'].str.strip().str.title()
cards_df['type'] = cards_df['type'].str.strip().str.title()
cards_df['archetype'] = cards_df['archetype'].str.strip().str.title()
print("Standardized text columns")


cards_df = cards_df[(cards_df['elixir_cost'] >= 1) & (cards_df['elixir_cost'] <= 9)]
print("Validated elixir cost range (1-9)")


cards_df = cards_df[(cards_df['win_rate'] >= 0) & (cards_df['win_rate'] <= 100)]
print("Validated win rate range (0-100%)")


cards_df = cards_df.reset_index(drop=True)

print(f"\nFinal cards count: {len(cards_df)}")
cards_df.head()

CLEANING CARDS DATASET
Removed duplicate cards: 0
Dropped rows with NaN values
Standardized text columns
Validated elixir cost range (1-9)
Validated win rate range (0-100%)

Final cards count: 121


,card_id,name,elixir_cost,rarity,type,arena_unlocked,damage,hitpoints,archetype,win_rate,usage_rate,popularity_score
0,1,Knight,3,Common,Troop,0,202,1766,Beatdown,54.25,24.51,36.41
1,2,Archers,3,Common,Troop,0,112,304,Control,51.66,7.19,24.98
2,3,Bomber,2,Common,Troop,2,225,304,Control,48.78,4.09,21.97
3,4,Goblins,2,Common,Troop,1,120,202,Cycle,45.35,4.46,20.82
4,5,Spear Goblins,2,Common,Troop,1,81,133,Cycle,48.61,3.83,21.74


In [6]:

print("ADDING NEW FEATURES TO CARDS")
print("=" * 50)

cards_df['damage_per_elixir'] = round(cards_df['damage'] / cards_df['elixir_cost'], 2)


cards_df['hp_per_elixir'] = round(cards_df['hitpoints'] / cards_df['elixir_cost'], 2)


cards_df['strength_score'] = round(
    (cards_df['win_rate'] * 0.5) +
    (cards_df['usage_rate'] * 0.3) +
    (cards_df['popularity_score'] * 0.2), 2
)


def cost_category(elixir):
    if elixir <= 2:
        return "Cheap"
    elif elixir <= 4:
        return "Medium"
    elif elixir <= 6:
        return "Expensive"
    else:
        return "Heavy"

cards_df['cost_category'] = cards_df['elixir_cost'].apply(cost_category)

print("Added: damage_per_elixir")
print("Added: hp_per_elixir")
print("Added: strength_score")
print("Added: cost_category")

cards_df.head()

ADDING NEW FEATURES TO CARDS
Added: damage_per_elixir
Added: hp_per_elixir
Added: strength_score
Added: cost_category


,card_id,name,elixir_cost,rarity,type,arena_unlocked,damage,hitpoints,archetype,win_rate,usage_rate,popularity_score,damage_per_elixir,hp_per_elixir,strength_score,cost_category
0,1,Knight,3,Common,Troop,0,202,1766,Beatdown,54.25,24.51,36.41,67.33,588.67,41.76,Medium
1,2,Archers,3,Common,Troop,0,112,304,Control,51.66,7.19,24.98,37.33,101.33,32.98,Medium
2,3,Bomber,2,Common,Troop,2,225,304,Control,48.78,4.09,21.97,112.50,152.00,30.01,Cheap
3,4,Goblins,2,Common,Troop,1,120,202,Cycle,45.35,4.46,20.82,60.00,101.00,28.18,Cheap
4,5,Spear Goblins,2,Common,Troop,1,81,133,Cycle,48.61,3.83,21.74,40.50,66.50,29.80,Cheap


In [7]:

print(" CLEANING DECKS DATASET")
print("=" * 50)

initial_rows = len(decks_df)

decks_df = decks_df.drop_duplicates()
print(f"Removed duplicate decks: {initial_rows - len(decks_df)}")


decks_df = decks_df.dropna()


decks_df = decks_df[(decks_df['avg_elixir'] >= 1) & (decks_df['avg_elixir'] <= 9)]
print(" Validated avg elixir range")

decks_df = decks_df[(decks_df['win_rate'] >= 0) & (decks_df['win_rate'] <= 100)]
print(" Validated win rate range")


def deck_quality(win_rate):
    if win_rate >= 60:
        return "Excellent"
    elif win_rate >= 50:
        return "Good"
    elif win_rate >= 45:
        return "Average"
    else:
        return "Weak"

decks_df['quality'] = decks_df['win_rate'].apply(deck_quality)
print("Added deck quality category")


decks_df = decks_df.reset_index(drop=True)

print(f"\n Final decks count: {len(decks_df)}")
decks_df.head()

 CLEANING DECKS DATASET
Removed duplicate decks: 0
 Validated avg elixir range
 Validated win rate range
Added deck quality category

 Final decks count: 480


,deck_id,deck_name,tier,source,card_1,card_2,card_3,card_4,card_5,card_6,card_7,card_8,avg_elixir,archetype,win_rate,usage_count,total_battles,quality
0,1,2.6 Hog Cycle,S,Classic competitive deck,Hog Rider,Musketeer,Ice Spirit,Skeletons,Fireball,The Log,Cannon,Ice Golem,2.62,Cycle,55.81,1273,9777,Good
1,2,2.6 Hog Cycle,S,Classic competitive deck,Hog Rider,Musketeer,Ice Spirit,Skeletons,Fireball,The Log,Cannon,Ice Golem,2.62,Cycle,57.39,1275,4444,Good
2,3,2.6 Hog Cycle,S,Classic competitive deck,Hog Rider,Musketeer,Ice Spirit,Skeletons,Fireball,The Log,Cannon,Ice Golem,2.62,Cycle,57.15,1338,4951,Good
3,4,2.6 Hog Cycle,S,Classic competitive deck,Hog Rider,Musketeer,Ice Spirit,Skeletons,Fireball,The Log,Cannon,Ice Golem,2.62,Cycle,56.85,1321,8855,Good
4,5,2.6 Hog Cycle,S,Classic competitive deck,Hog Rider,Musketeer,Ice Spirit,Skeletons,Fireball,The Log,Cannon,Ice Golem,2.62,Cycle,57.39,1295,6212,Good


In [8]:

print("CLEANING BATTLES DATASET")
print("=" * 50)

initial_rows = len(battles_df)


battles_df = battles_df.drop_duplicates()
print(f" Removed duplicate battles: {initial_rows - len(battles_df)}")


battles_df = battles_df.dropna()


battles_df = battles_df[
    (battles_df['player_1_trophies'] >= 0) & (battles_df['player_1_trophies'] <= 10000) &
    (battles_df['player_2_trophies'] >= 0) & (battles_df['player_2_trophies'] <= 10000)
]
print(" Validated trophy ranges")


battles_df = battles_df[
    (battles_df['player_1_crowns'].between(0, 3)) &
    (battles_df['player_2_crowns'].between(0, 3))
]
print(" Validated crown counts (0-3)")


battles_df['trophy_difference'] = abs(
    battles_df['player_1_trophies'] - battles_df['player_2_trophies']
)


battles_df['battle_intensity'] = (
    battles_df['player_1_crowns'] + battles_df['player_2_crowns']
)


def match_type(diff):
    if diff < 100:
        return "Balanced"
    elif diff < 500:
        return "Slight Mismatch"
    else:
        return "Heavy Mismatch"

battles_df['match_type'] = battles_df['trophy_difference'].apply(match_type)
print(" Added: trophy_difference, battle_intensity, match_type")


battles_df = battles_df.reset_index(drop=True)

print(f"\n Final battles count: {len(battles_df)}")
battles_df.head()

CLEANING BATTLES DATASET
 Removed duplicate battles: 0
 Validated trophy ranges
 Validated crown counts (0-3)
 Added: trophy_difference, battle_intensity, match_type

 Final battles count: 5000


,battle_id,player_1_deck,player_1_elixir,player_1_trophies,player_1_crowns,player_2_deck,player_2_elixir,player_2_trophies,player_2_crowns,winner,battle_duration_sec,game_mode,trophy_difference,battle_intensity,match_type
0,1,Lava Hound|Inferno Dragon|Balloon|Mega Minion|...,4.00,4455,2,Balloon|Miner|Inferno Dragon|Skeleton Army|Fir...,3.12,4471,0,player_1,122,Ladder,16,2,Balanced
1,2,Mega Knight|Goblin Barrel|Princess|Goblin Gang...,4.00,6301,3,X-Bow|Tesla|Archers|Knight|Skeletons|Ice Spiri...,3.00,6423,1,player_1,225,Tournament,122,4,Slight Mismatch
2,3,PEKKA|Wizard|Battle Ram|Bandit|Magic Archer|Po...,4.00,5462,0,X-Bow|Knight|Tesla|Archers|Skeletons|Ice Spiri...,3.00,5404,3,player_2,85,Tournament,58,3,Balanced
3,4,Skeleton King|Battle Ram|Bandit|Magic Archer|E...,3.50,4344,0,Lava Hound|Inferno Dragon|Balloon|Mega Minion|...,4.00,4515,0,draw,110,Ladder,171,0,Slight Mismatch
4,5,Executioner|Tornado|Knight|Skeletons|Fireball|...,2.75,7373,0,Royal Champion|Royal Recruits|Mighty Miner|Hun...,3.88,7409,3,player_2,229,Ladder,36,3,Balanced


In [9]:

print(" CLEANING META DATASET")
print("=" * 50)

initial_rows = len(meta_df)

meta_df = meta_df.drop_duplicates()
print(f" Removed duplicates: {initial_rows - len(meta_df)}")


meta_df = meta_df.dropna()


meta_df['card_name'] = meta_df['card_name'].str.strip()
meta_df['season'] = meta_df['season'].str.strip()
meta_df['trend'] = meta_df['trend'].str.strip().str.title()


meta_df = meta_df[(meta_df['win_rate'] >= 0) & (meta_df['win_rate'] <= 100)]
meta_df = meta_df[(meta_df['usage_rate'] >= 0) & (meta_df['usage_rate'] <= 100)]


meta_df = meta_df.reset_index(drop=True)

print(f"\n Final meta records: {len(meta_df)}")
meta_df.head()

 CLEANING META DATASET
 Removed duplicates: 0

 Final meta records: 635


,card_name,season,usage_rate,win_rate,ranking,trend
0,Knight,Season_45,25.15,53.77,99,Stable
1,Knight,Season_46,23.71,53.78,39,Stable
2,Knight,Season_47,24.36,54.78,54,Stable
3,Knight,Season_48,25.25,54.04,93,Stable
4,Knight,Season_49,24.40,53.74,21,Stable


In [10]:

print(" MISSING VALUES (AFTER CLEANING)")
print("=" * 50)

cleaned_datasets = {
    "Cards":   cards_df,
    "Decks":   decks_df,
    "Battles": battles_df,
    "Meta":    meta_df
}

for name, df in cleaned_datasets.items():
    missing = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()
    print(f" {name:10} → Missing: {missing} | Duplicates: {duplicates} | Rows: {len(df)}")

 MISSING VALUES (AFTER CLEANING)
 Cards      → Missing: 0 | Duplicates: 0 | Rows: 121
 Decks      → Missing: 0 | Duplicates: 0 | Rows: 480
 Battles    → Missing: 0 | Duplicates: 0 | Rows: 5000
 Meta       → Missing: 0 | Duplicates: 0 | Rows: 635


In [11]:

print(" SAVING CLEANED DATASETS")
print("=" * 50)

cards_df.to_csv(PROCESSED_PATH / "cards_cleaned.csv", index=False)
print(" Saved: cards_cleaned.csv")

decks_df.to_csv(PROCESSED_PATH / "decks_cleaned.csv", index=False)
print(" Saved: decks_cleaned.csv")

battles_df.to_csv(PROCESSED_PATH / "battles_cleaned.csv", index=False)
print(" Saved: battles_cleaned.csv")

meta_df.to_csv(PROCESSED_PATH / "meta_cleaned.csv", index=False)
print(" Saved: meta_cleaned.csv")

print(f"\n All cleaned files saved in: {PROCESSED_PATH.absolute()}")

 SAVING CLEANED DATASETS
 Saved: cards_cleaned.csv
 Saved: decks_cleaned.csv
 Saved: battles_cleaned.csv
 Saved: meta_cleaned.csv

 All cleaned files saved in: e:\Clash Royale\notebooks\..\dataset\processed


In [12]:
#
print("=" * 70)
print(" DATA CLEANING SUMMARY REPORT")
print("=" * 70)

print(f"""
 CARDS DATASET
   Total cards:      {len(cards_df)}
   New features:     damage_per_elixir, hp_per_elixir, strength_score, cost_category
   Rarities:         {cards_df['rarity'].nunique()}
   Cost categories:  {cards_df['cost_category'].nunique()}

 DECKS DATASET
   Total decks:      {len(decks_df)}
   New features:     quality
   Excellent decks:  {(decks_df['quality'] == 'Excellent').sum()}
   Avg win rate:     {decks_df['win_rate'].mean():.2f}%

  BATTLES DATASET
   Total battles:    {len(battles_df)}
   New features:     trophy_difference, battle_intensity, match_type
   Balanced matches: {(battles_df['match_type'] == 'Balanced').sum()}

 META DATASET
   Total records:    {len(meta_df)}
   Seasons covered:  {meta_df['season'].nunique()}

 NEXT STEP: Notebook 03 - Exploratory Data Analysis (EDA) with Visualizations!
""")

 DATA CLEANING SUMMARY REPORT

 CARDS DATASET
   Total cards:      121
   New features:     damage_per_elixir, hp_per_elixir, strength_score, cost_category
   Rarities:         5
   Cost categories:  4

 DECKS DATASET
   Total decks:      480
   New features:     quality
   Excellent decks:  0
   Avg win rate:     52.48%

  BATTLES DATASET
   Total battles:    5000
   New features:     trophy_difference, battle_intensity, match_type
   Balanced matches: 2132

 META DATASET
   Total records:    635
   Seasons covered:  5

 NEXT STEP: Notebook 03 - Exploratory Data Analysis (EDA) with Visualizations!

